In [ ]:
"""
XRD Plotting Utility — Masked / Naked Mode with Reference Markers
==================================================================
Loads one or more .xy diffractograms (two-column: 2θ [deg], intensity [a.u.]),
and produces publication-ready figures with:

  - log₁₀ intensity axis (XRD standard; switchable to linear)
  - Optional angular masks to suppress known artefact / substrate peaks
  - Colour-coded vertical reference markers (FePt, Si, SiO₂, Fe–O phases)
    with a per-line legend showing (hkl) + 2θ value
  - Global maximum peak annotation (optional)
  - Export as PNG (600 dpi) + SVG per file

Modes
-----
  masked : apply the angular masks defined in MASKS below, then plot
  naked  : no masking — raw data as measured

Workflow (GUI-driven):
  1. Run the cell → file dialog opens for .xy file(s)
  2. Terminal prompt: choose masked or naked
  3. Outputs saved to `xrd_outputs/{base}_{mode}_{timestamp}.png/.svg`

Inputs  : .xy files (whitespace- or comma-separated, comment lines ignored).
Outputs : PNG + SVG per file, saved to a timestamped subfolder in `xrd_outputs/`.
"""

from __future__ import annotations

import argparse
import glob
import os
from dataclasses import dataclass
from datetime import datetime
from tkinter import Tk, filedialog
from typing import Dict, Iterable, List, Tuple

import matplotlib.pyplot as plt
import numpy as np


# ===========================================================================
# 1. SETTINGS — edit these to match your experiment
# ===========================================================================

OUTDIR_BASE = "xrd_outputs"

# --- Angular masks (2θ in degrees) -----------------------------------------
# Add MaskRange entries to suppress known substrate peaks or setup artefacts.
# These are only applied in "masked" mode.
MASKS: List["MaskRange"] = [
    MaskRange(68.8, 69.4, "Si (400)"),   # dominant Si substrate peak
    # MaskRange(XX.X, YY.Y, "label"),    # add more as needed
]

# --- Reference marker groups to show ---------------------------------------
# Set to True/False to toggle each group independently.
SHOW_FEPT      = True   # FePt L1₀ reference peaks (orange)
SHOW_SI        = True   # Si substrate peaks (gray)
SHOW_SIO2      = True   # SiO₂ amorphous hump (blue)
SHOW_MAGNETITE = True   # Fe₃O₄ magnetite candidates (green, dotted)
SHOW_HEMATITE  = False  # α-Fe₂O₃ hematite candidates (green, dashed) — often overlapping
SHOW_MAGHEMITE = False  # γ-Fe₂O₃ maghemite candidates (green, dash-dot)


# ===========================================================================
# 2. MASK CONFIGURATION
# ===========================================================================

@dataclass(frozen=True)
class MaskRange:
    """Angular range to exclude from the masked plot."""
    start_deg: float
    end_deg:   float
    label:     str = ""


# ===========================================================================
# 3. REFERENCE PEAKS & STYLES
# ===========================================================================

def reference_peaks() -> Dict[str, Dict[str, float]]:
    """
    Reference 2θ positions (Cu Kα, λ = 1.5406 Å) for guide markers.
    Values are nominal; strain / composition / instrument offset can shift peaks.
    """
    return {
        "FePt": {
            "(001)": 23.20,
            "(111)": 40.00,
            "(110)": 41.20,
            "(002)": 47.50,
        },
        "Si": {
            "(111)": 28.40,
            "(220)": 47.30,
            "(311)": 56.10,
            "(400)": 69.10,
        },
        "SiO2": {
            "amorphous hump": 22.00,
        },
        # Oxygen-containing phase candidates — useful for identifying
        # oxidation in as-deposited or poorly annealed FePt films
        "Fe–O (magnetite)": {
            "(220)": 30.10,
            "(311)": 35.45,
            "(400)": 43.10,
            "(422)": 53.45,
            "(511)": 57.00,
            "(440)": 62.60,
        },
        "Fe–O (hematite)": {
            "(104)": 33.20,
            "(110)": 35.60,
            "(024)": 49.50,
            "(116)": 54.10,
            "(214)": 62.40,
        },
        "Fe–O (maghemite)": {
            "(311)": 35.60,
            "(400)": 43.30,
            "(511)": 57.20,
            "(440)": 62.90,
        },
    }


def reference_styles() -> Dict[str, Dict[str, object]]:
    """
    Line style per reference group.
    FePt=orange, Si=gray, SiO₂=blue, Fe–O phases=green (different linestyles).
    """
    return {
        "FePt":             {"color": "tab:orange", "linestyle": "--",  "linewidth": 1.0, "alpha": 0.85},
        "Si":               {"color": "0.35",       "linestyle": "--",  "linewidth": 0.9, "alpha": 0.75},
        "SiO2":             {"color": "tab:blue",   "linestyle": "--",  "linewidth": 0.9, "alpha": 0.75},
        "Fe–O (magnetite)": {"color": "tab:green",  "linestyle": ":",   "linewidth": 1.1, "alpha": 0.85},
        "Fe–O (hematite)":  {"color": "tab:green",  "linestyle": "--",  "linewidth": 0.9, "alpha": 0.55},
        "Fe–O (maghemite)": {"color": "tab:green",  "linestyle": "-.",  "linewidth": 0.9, "alpha": 0.55},
    }


def _group_enabled(material: str) -> bool:
    """Return True if this reference group is toggled on in SETTINGS."""
    return {
        "FePt":             SHOW_FEPT,
        "Si":               SHOW_SI,
        "SiO2":             SHOW_SIO2,
        "Fe–O (magnetite)": SHOW_MAGNETITE,
        "Fe–O (hematite)":  SHOW_HEMATITE,
        "Fe–O (maghemite)": SHOW_MAGHEMITE,
    }.get(material, True)


def draw_reference_lines(ax: plt.Axes, x_min: float, x_max: float) -> None:
    """
    Draw vertical reference lines for all enabled groups within the plotted
    2θ range. Legend entries list material, (hkl), and 2θ value — placed
    outside the axes on the right so it never overlaps the data.
    """
    peaks  = reference_peaks()
    styles = reference_styles()

    handles: List[plt.Line2D] = []
    labels:  List[str]        = []

    for material, pk in peaks.items():
        if not _group_enabled(material):
            continue
        st = styles.get(material, {"linestyle": "--", "linewidth": 0.8, "alpha": 0.6})
        for hkl, theta in pk.items():
            if theta < x_min or theta > x_max:
                continue
            line = ax.axvline(theta, **st)
            handles.append(line)
            prefix = "~" if material == "SiO2" else ""
            labels.append(f"{material} {hkl} — {prefix}{theta:.2f}°")

    if handles:
        ax.legend(
            handles, labels,
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            borderaxespad=0.0,
            frameon=True,
            fontsize=9,
            handlelength=3,
            labelspacing=0.6,
        )


# ===========================================================================
# 4. DATA LOADING & PROCESSING
# ===========================================================================

def load_xy(path: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load a two-column .xy file (2θ [deg], intensity [a.u.]).
    Tolerates whitespace or comma separation; ignores comment lines.
    Data is sorted by 2θ to handle any non-monotonic exports.
    """
    data: List[Tuple[float, float]] = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(("#", ";", "//")):
                continue
            parts = s.replace(",", " ").split()
            if len(parts) < 2:
                continue
            try:
                data.append((float(parts[0]), float(parts[1])))
            except ValueError:
                continue

    if not data:
        raise ValueError(f"No numeric two-column data found in: {path}")

    arr = np.array(data, dtype=float)
    idx = np.argsort(arr[:, 0])
    return arr[idx, 0], arr[idx, 1]


def apply_masks(
    two_theta: np.ndarray,
    intensity: np.ndarray,
    masks: Iterable[MaskRange],
) -> Tuple[np.ndarray, np.ndarray]:
    """Remove data points within the specified angular mask ranges."""
    keep = np.ones_like(two_theta, dtype=bool)
    for m in masks:
        keep &= ~((two_theta >= m.start_deg) & (two_theta <= m.end_deg))
    return two_theta[keep], intensity[keep]


def safe_log10(y: np.ndarray) -> np.ndarray:
    """
    Log₁₀ scaling with numerical safety — clips non-positive values to the
    smallest positive value present (or 1e-12 if none exist).
    """
    y     = np.asarray(y, dtype=float)
    y_pos = y[np.isfinite(y) & (y > 0)]
    a_min = float(np.nanmin(y_pos)) if y_pos.size else 1e-12
    return np.log10(np.clip(y, a_min=a_min, a_max=None))


# ===========================================================================
# 5. PLOTTING & EXPORT
# ===========================================================================

def plot_xrd(
    two_theta:       np.ndarray,
    intensity:       np.ndarray,
    title:           str,
    log_scale:       bool,
    annotate_max:    bool,
    out_png:         str,
    out_svg:         str,
    show_references: bool,
) -> None:
    """
    Produce and save a single XRD figure.
    Wide figure (14 × 6 in) with room on the right for the reference legend.
    """
    fig, ax = plt.subplots(figsize=(14, 6))

    y_plot = safe_log10(intensity) if log_scale else intensity
    ax.plot(two_theta, y_plot, lw=1.0, color='tab:blue')

    ax.set_xlabel(r"2$\theta$ (deg)", fontsize=12)
    ax.set_ylabel(
        r"$\log_{10}$(Intensity) (a.u.)" if log_scale else "Intensity (a.u.)",
        fontsize=12
    )
    ax.set_title(title, fontsize=11)
    ax.grid(True, linewidth=0.4, alpha=0.5)

    if annotate_max:
        x_max, y_max_raw = two_theta[np.nanargmax(intensity)], np.nanmax(intensity)
        y_ann = float(np.log10(max(y_max_raw, 1e-12))) if log_scale else float(y_max_raw)
        ax.scatter([x_max], [y_ann], marker="o", zorder=5)
        ax.annotate(
            f"Max @ {x_max:.3f}°",
            xy=(x_max, y_ann),
            xytext=(10, 10),
            textcoords="offset points",
            fontsize=9,
        )

    if show_references:
        x_min = float(np.nanmin(two_theta))
        x_max = float(np.nanmax(two_theta))
        draw_reference_lines(ax, x_min=x_min, x_max=x_max)

    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    # Reserve space on the right for the external reference legend
    fig.tight_layout(rect=[0, 0, 0.82, 1])
    fig.savefig(out_png, dpi=600)
    fig.savefig(out_svg)
    plt.close(fig)


# ===========================================================================
# 6. FILE DISCOVERY & INTERACTIVE HELPERS
# ===========================================================================

def pick_xy_files() -> List[str]:
    """Open a GUI dialog for selecting one or more .xy files."""
    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    files = filedialog.askopenfilenames(
        title="Select XRD .xy file(s)",
        filetypes=[("XRD files", "*.xy"), ("All files", "*.*")]
    )
    root.destroy()
    return list(files)


def ask_mode_interactive() -> str:
    """Prompt the user to choose masked or naked mode."""
    print("\nSelect XRD plotting mode:")
    print("  [1] Masked  — apply angular masks defined in MASKS")
    print("  [2] Naked   — raw data, no masking")
    while True:
        choice = input("Selection (1/2): ").strip()
        if choice == "1":
            return "masked"
        if choice == "2":
            return "naked"
        print("  Invalid — enter 1 or 2.")


# ===========================================================================
# 7. MAIN
# ===========================================================================

def main() -> None:
    parser = argparse.ArgumentParser(description="Plot XRD .xy files.")
    parser.add_argument("--input",   default=None,
                        help="Path to .xy file, directory, or glob. Omit for GUI dialog.")
    parser.add_argument("--output",  default=OUTDIR_BASE,
                        help=f"Output directory (default: {OUTDIR_BASE}).")
    parser.add_argument("--mode",    choices=["masked", "naked", "ask"], default="ask")
    parser.add_argument("--linear",  action="store_true",
                        help="Linear intensity axis (default: log₁₀).")
    parser.add_argument("--annotate-max", action="store_true",
                        help="Annotate the global maximum peak position.")
    parser.add_argument("--title",   default="",
                        help="Figure title. Defaults to filename stem if empty.")
    parser.add_argument("--no-references", action="store_true",
                        help="Suppress all reference markers.")
    # Jupyter-safe: ignore unknown kernel args
    args, _ = parser.parse_known_args()

    # --- File selection ---
    if args.input is None:
        files = pick_xy_files()
        if not files:
            raise SystemExit("No files selected.")
    elif os.path.isdir(args.input):
        files = sorted(glob.glob(os.path.join(args.input, "*.xy")))
    else:
        files = sorted(glob.glob(args.input)) or ([args.input] if os.path.isfile(args.input) else [])
    if not files:
        raise FileNotFoundError(f"No .xy files found for: {args.input}")

    # --- Mode selection ---
    mode = args.mode if args.mode != "ask" else ask_mode_interactive()

    print(f"\nFiles   : {len(files)}")
    print(f"Mode    : {mode}")
    print(f"Y-scale : {'linear' if args.linear else 'log₁₀'}")
    if mode == "masked":
        print(f"Masks   : {len(MASKS)} range(s)")
    print()

    # --- Process each file ---
    for fp in files:
        base = os.path.splitext(os.path.basename(fp))[0]
        ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_base = os.path.join(args.output, f"{base}_{mode}_{ts}")

        two_theta, intensity = load_xy(fp)

        if mode == "masked" and MASKS:
            two_theta, intensity = apply_masks(two_theta, intensity, MASKS)

        title = args.title.strip() or base

        plot_xrd(
            two_theta       = two_theta,
            intensity       = intensity,
            title           = title,
            log_scale       = not args.linear,
            annotate_max    = args.annotate_max,
            out_png         = out_base + ".png",
            out_svg         = out_base + ".svg",
            show_references = not args.no_references,
        )
        print(f"Saved: {out_base}.png / .svg")

    print("\nDONE.")


if __name__ == "__main__":
    main()
